# 01 — Autoencoder supervisado de MAG

**Proyecto:** detección y localización temporal de cruces MPB en Marte, con MAVEN.

Este cuaderno se construye **únicamente para datos reales**. No usa ni genera datos artificiales y **no se ha ejecutado**. Los resultados y las figuras quedarán vacíos hasta que estén disponibles los archivos finales de normalización y habilitemos explícitamente una consulta.

Objetivo del futuro modelo: recibir una ventana temporal de 600 s de campo magnético de MAVEN, reconstruirla mediante un decoder y, mediante una rama supervisada adicional, predecir la cercanía del centro de la ventana a un cruce MPB etiquetado.

**Entrada:** 4 canales: Bx, By, Bz y |B|.

**Salida de detección:** puntaje entre 0 y 1 asociado al objetivo gaussiano; **no es una probabilidad calibrada ni una anomalía de reconstrucción**.

## 1. Preparar el entorno

- Trabajaremos dentro de .venv desde la raíz del repositorio.
- Cada experimento se identifica por un **fold de validación**. Los ejemplos del notebook se seleccionarán exclusivamente entre los otros tres folds (TRAIN).
- El normalizador debe ser el archivo **final** del instrumento y fold correspondiente. No se permite usar estadísticas del fold de validación ni checkpoints parciales.
- No guardaremos mediciones de MAVEN en archivos locales; sólo conservaríamos más adelante parámetros del modelo, métricas y figuras.

**Estado:** las celdas tienen el contador de ejecución vacío; ningún entrenamiento ni consulta fue realizada al crear este notebook.

In [ ]:
from pathlib import Path
from datetime import timedelta
from importlib import import_module
import json
import sys

import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().resolve()
if not (repo / "scripts").is_dir():
    repo = repo.parent
if not (repo / "scripts").is_dir():
    raise RuntimeError("Abrí Jupyter desde la raíz del repositorio o notebooks/")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

bq = import_module("scripts.1_bigquery.1c_bigquery_event_pilot")
windowing = import_module("scripts.3_windowing.3a_windowing")
window_pilot = import_module("scripts.3_windowing.3b_bigquery_window_pilot")
batching = import_module("scripts.4_preprocessing.4a_native_batches")
preprocess_pilot = import_module("scripts.4_preprocessing.4b_fold_preprocessing_pilot")
fold_fitter = import_module("scripts.4_preprocessing.4c_fit_fold_normalizers")

INSTRUMENT = "MAG"
VALIDATION_FOLD = 1  # luego repetiremos para 2, 3 y 4
MANIFEST = repo / "data/splits/cv4_monthly_manifest.csv"
NORMALIZER = repo / (
    f"results/metrics/fold_normalization/"
    f"fold_{VALIDATION_FOLD:02d}_mag_normalizer.json"
)

# BLOQUEO INTENCIONAL: cambiar a True sólo cuando queramos consultar MAVEN real.
EXECUTE_REAL_BIGQUERY = False
MAX_QUERY_GIB = 1.0
MAX_RETURNED_ROWS = 25000
print("Notebook preparado para", INSTRUMENT, "| Fold", VALIDATION_FOLD)
print("Consultas habilitadas:", EXECUTE_REAL_BIGQUERY)

## 2. Cargar exclusivamente el normalizador completo

El archivo final fold_XX_mag_normalizer.json contiene las estadísticas de las observaciones de entrenamiento, calculadas una sola vez por instante temporal.

**Controles incluidos:** instrumento, fold, firma del manifiesto, canales, valores finitos y escalas positivas. Para SWEA también se verifica que cada canal corresponda a una energía física en la banda acordada.

En este notebook nunca usamos *_checkpoint.json como normalizador definitivo.

In [ ]:
def load_finished_fold_normalizer(path, manifest, fold, instrument):
    if not path.is_file():
        raise FileNotFoundError(
            f"Aún falta el normalizador FINAL: {path}. "
            "Esperar a que termine el paso 4c."
        )
    info = json.loads(path.read_text(encoding="utf-8"))
    if info.get("schema") != 1:
        raise ValueError("Formato del normalizador no reconocido.")
    if info.get("held_out_fold") != fold or info.get("instrument") != instrument:
        raise ValueError("El normalizador no corresponde a este fold/instrumento.")
    if int(info.get("n_unique_train_epochs", 0)) < 2:
        raise ValueError("El normalizador no tiene suficientes muestras.")

    # Auditar que la asignación actual de folds sea la usada al normalizar.
    records = window_pilot.read_manifest(manifest)
    intervals, _, _ = fold_fitter.plan_training_days(records, fold, instrument)
    expected_signature = fold_fitter.planned_signature(
        manifest, fold, instrument, intervals
    )
    if info.get("signature") != expected_signature:
        raise ValueError("El normalizador fue generado con otra configuración/etiquetas.")

    mean = np.asarray(info["mean"], dtype=np.float64)
    scale = np.asarray(info["scale"], dtype=np.float64)
    if (mean.ndim != 1 or mean.shape != scale.shape
            or not np.isfinite(mean).all() or not np.isfinite(scale).all()
            or np.any(scale <= 0)):
        raise ValueError("Media/escala del normalizador inválidas.")
    if instrument == "MAG" and mean.size != 4:
        raise ValueError("MAG debe tener exactamente 4 canales.")
    energies = None
    if instrument == "SWEA":
        energies = np.asarray(info["swea_selected_energies_ev"], dtype=float)
        if (energies.shape != mean.shape or not np.isfinite(energies).all()
                or np.any(energies < 30) or np.any(energies > 100)):
            raise ValueError("Energías SWEA distintas de las acordadas.")

    # Restaurar solamente las estadísticas finales, sin volver a entrenarlas.
    normalizer = batching.FoldChannelNormalizer(instrument, held_out_fold=fold)
    normalizer.count = int(info["n_unique_train_epochs"])
    normalizer.n_features = len(mean)
    normalizer.mean = mean
    normalizer.scale = scale
    normalizer.is_frozen = True
    return normalizer, energies

# No se carga el archivo hasta que se habilite el trabajo con datos reales.

## 3. Seleccionar un evento real de ENTRENAMIENTO

Elegiremos un evento cuyo fold sea **distinto** del reservado para validación. Consultaremos sólo un intervalo de ±35 min (70 min) alrededor de su etiqueta MPB; eso permite generar ventanas de ±5 min para centros dentro de ±30 min.

La consulta hace primero una estimación de bytes. Se detendrá si supera el tope configurado. No realizaremos una consulta por cada ventana, y las mediciones existirán únicamente en RAM.

In [ ]:
def prepare_one_training_event(*, fold, instrument, max_query_gib, max_rows):
    from google.cloud import bigquery

    normalizer, normalizer_energies = load_finished_fold_normalizer(
        NORMALIZER, MANIFEST, fold, instrument
    )
    records = window_pilot.read_manifest(MANIFEST)
    training = [
        row for row in records
        if row["fold"] != fold and row[f"{instrument.lower()}_available"] == "1"
    ]
    if not training:
        raise ValueError("No hay eventos TRAIN disponibles.")
    event = training[0]  # elección determinista, cambiable más adelante
    training_labels = np.asarray([
        row["time"].timestamp() for row in records if row["fold"] != fold
    ], dtype=np.float64)

    start = event["time"] - timedelta(minutes=35)
    end = event["time"] + timedelta(minutes=35)
    sql = (bq.MAG_SQL.format(mag=bq.MAG)
           if instrument == "MAG" else
           bq.SWEA_SQL.format(swea=bq.SWEA, axes=bq.AXES))

    client = bigquery.Client(project=bq.PROJECT)
    bq.validate_schema(client)
    estimate_job = client.query(
        sql, job_config=bq.query_config(start, end, dry_run=True, max_bytes=1)
    )
    estimated_bytes = int(estimate_job.total_bytes_processed or 0)
    print("Evento TRAIN:", event["time"].isoformat())
    print("Estimación BigQuery:", round(estimated_bytes / bq.GIB, 5), "GiB")
    if estimated_bytes > max_query_gib * bq.GIB:
        raise RuntimeError("El costo estimado supera MAX_QUERY_GIB.")

    plan = (event, instrument, sql, start, end, estimated_bytes)
    times, values, energies = preprocess_pilot.fetch(
        client, plan, int(max_query_gib * bq.GIB), max_rows
    )
    if not len(times):
        raise ValueError("El evento no tiene mediciones.")
    if instrument == "SWEA" and (
        energies.shape != normalizer_energies.shape
        or not np.allclose(energies, normalizer_energies, rtol=1e-6, atol=1e-6)
    ):
        raise ValueError("La grilla de energías SWEA difiere de la normalización.")

    stats, windows = preprocess_pilot.window_sample(
        times, values, event, training_labels, sigma=60.0
    )
    batch = batching.collate_native_windows(windows, normalizer=normalizer)
    return {"event": event, "stats": stats, "windows": windows,
            "batch": batch, "energies": energies}

prepared = None
if not EXECUTE_REAL_BIGQUERY:
    print("Bloqueado: esperando normalización final y decisión de cargar MAVEN real.")
else:
    prepared = prepare_one_training_event(
        fold=VALIDATION_FOLD,
        instrument=INSTRUMENT,
        max_query_gib=MAX_QUERY_GIB,
        max_rows=MAX_RETURNED_ROWS,
    )
    print("Batch:", tuple(prepared["batch"].x.shape))
    print("Targets:", prepared["batch"].y.flatten().tolist())

## 4. Inspección visual de una ventana real

Cuando habilitemos la consulta, graficaremos una ventana elegida de la señal física **antes de normalizar**, para entender qué medirá cada canal.

**No se genera ninguna figura hasta que se obtenga una ventana MAVEN real.**

In [ ]:
if prepared is None:
    print("No hay datos cargados. Por ahora no se generan figuras.")
else:
    raw_window = min(prepared["windows"], key=lambda w: abs(w.offset_from_nearest_mpb_s))
    times = raw_window.relative_times_s
    fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    for i, title in enumerate(["Bx", "By", "Bz", "|B|"]):
        axes[i].plot(times, raw_window.data[:, i], linewidth=1)
        axes[i].set_ylabel(f"{title} (nT)")
        axes[i].grid(alpha=0.25)
    axes[-1].set_xlabel("Segundos respecto al centro de la ventana")
    fig.suptitle("Ventana MAG real (no simulada)")
    fig.tight_layout()
    plt.show()

## 5. Primera arquitectura MAG

Proponemos un encoder convolucional temporal que recibe un lote (B,4,T), un decoder que reconstruye los cuatro canales y un detector supervisado que predice cercanía temporal al cruce etiquetado. Vamos a verificar primero las dimensiones usando datos MAVEN reales. La arquitectura es provisional; todavía no entrenamos.